In [222]:
import pandas as pd

In [223]:
df = pd.read_csv("Cleaned_2022.csv")

In [224]:
df.head()

,Unnamed: 0,RANK,Country,Happiness score,Whisker-high,Whisker-low,Dystopia (1.83) + residual,Explained by: GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption,GDP_tier,Computed_Rank
0,0,1,Finland,7821,7886,7756,2518,1892,1258,775,736,109,534,High,1
1,1,2,Denmark,7636,7710,7563,2226,1953,1243,777,719,188,532,High,2
2,2,3,Iceland,7557,7651,7464,2320,1936,1320,803,718,270,191,High,3
3,3,4,Switzerland,7512,7586,7437,2153,2026,1226,822,677,147,461,High,4
4,4,5,Netherlands,7415,7471,7359,2137,1945,1206,787,651,271,419,High,5


In [225]:
df = df.drop('Unnamed: 0', axis = 1)

## Q1: Compute the mean, median, and standard deviation of Happiness score.

In [227]:
print(df['Happiness score'].mean())
print(df['Happiness score'].median())
print(df['Happiness score'].std())

5553.575342465753
5568.5
1086.842607236694


## Q2: Find the skewness and kurtosis of Happiness score. Interpret the results.

In [229]:
print(round(df['Happiness score'].skew(),2))
print(round(df['Happiness score'].kurt(),2))

-0.25
-0.21


* The data is almost symmetric, with a small lean toward lower values.
* The distribution has slightly lighter tails (less extreme values) than a normal distribution.

## Q3: Which country has the highest and lowest Explained by: Healthy life expectancy?

In [232]:
highest_healthy = df.loc[df['Explained by: Healthy life expectancy'].idxmax(), 'Country']
highest_healthy

'Hong Kong S.A.R. of China'

In [233]:
lowest_healthy = df.loc[df['Explained by: Healthy life expectancy'].idxmin(), 'Country']
lowest_healthy

'Lesotho*'

## Q4: Create a correlation matrix between Happiness score, Explained by: GDP per capita, and Explained by: Social support.

* Which factor correlates most strongly with happiness?

In [235]:
corr_matrix = df[['Happiness score', 'Explained by: GDP per capita', 'Explained by: Social support']].corr()
corr_matrix

,Happiness score,Explained by: GDP per capita,Explained by: Social support
Happiness score,1.000000,0.763677,0.777889
Explained by: GDP per capita,0.763677,1.000000,0.722421
Explained by: Social support,0.777889,0.722421,1.000000


In [236]:
happiness_corr = corr_matrix['Happiness score'].drop('Happiness score')
top_factor = happiness_corr.idxmax()
top_factor

'Explained by: Social support'

## Q5: Assume Happiness score follows a normal distribution.

* Find the probability that a randomly chosen country has a score above 7.0.

* Find the z-score for India’s Happiness score.

In [238]:
from scipy.stats import norm

In [239]:
mean_score = df["Happiness score"].mean()
std_score = df["Happiness score"].std()

In [240]:
x = 7
z = (x-mean_score)/std_score

prob = 1 - norm.cdf(z)
print("Probability above 7.0:", prob)

Probability above 7.0: 0.9999998331836132


In [241]:
india_score = df.loc[df['Country'] == 'India', 'Happiness score'].values[0]
india_score

3777

In [242]:
z_ind = (india_score - mean_score)/std_score
print("Z-score for India:", z_ind)

Z-score for India: -1.6346206255041016


## Q6: Find the Interquartile Range (IQR) of the Happiness score and identify potential outlier countries (using the 1.5 × IQR rule).

* Compute Q1 (25th percentile) and Q3 (75th percentile) of Happiness score.

* Calculate IQR = Q3 – Q1.

* Define lower bound = Q1 – 1.5 × IQR, upper bound = Q3 + 1.5 × IQR.

* List all countries with Happiness scores outside these bounds.

In [244]:
Q1 = df['Happiness score'].quantile(0.25)
Q3 = df['Happiness score'].quantile(0.75)
IQR = Q3 - Q1
lw = (Q1-1.5*IQR) 
uw = (Q3 + 1.5*IQR)
outliers = (df['Happiness score'] < lw) | (df['Happiness score'] > uw)
print(df[outliers]['Country'])

145    Afghanistan
Name: Country, dtype: object


## Q7: Perform a t-test: Compare the mean Happiness score of the top 20 ranked countries vs the bottom 20 ranked countries.

* Is the difference statistically significant?

In [246]:
from scipy import stats

In [247]:
top_20 = df.nsmallest(20, 'RANK')
bottom_20 = df.nlargest(20, 'RANK')

In [248]:
top_scores = top_20['Happiness score']
bottom_scores = bottom_20['Happiness score']

In [249]:
t_stat, p_val = stats.ttest_ind(top_scores, bottom_scores)
print("T-statistic:", t_stat)
print("P-value:", p_val)

alpha = 0.05
if p_val < alpha:
    print("Reject H0 → Significant difference in mean Happiness scores.")
else:
    print("Fail to reject H0 → No significant difference.")

T-statistic: 25.055511657147136
P-value: 3.1183073125771913e-25
Reject H0 → Significant difference in mean Happiness scores.


## Q8: Perform a one-way ANOVA: Test whether the mean Happiness score differs significantly across the three GDP tiers you created (Low, Medium, High).

In [251]:
low_tier_happiness = df[df['GDP_tier'] == 'Low']['Happiness score']
medium_tier_happiness = df[df['GDP_tier'] == 'Medium']['Happiness score']
high_tier_happiness = df[df['GDP_tier'] == 'High']['Happiness score']

In [252]:
f_stat, p_val = stats.f_oneway(low_tier_happiness, medium_tier_happiness, high_tier_happiness)
print("F-statistic:", f_stat)
print("P-value:", p_val)

alpha = 0.05
if p_val < alpha:
    print("Reject H0 → Significant difference in mean Happiness scores.")
else:
    print("Fail to reject H0 → No significant difference.")

F-statistic: 81.83247478016695
P-value: 2.0419288489387768e-24
Reject H0 → Significant difference in mean Happiness scores.


## Q9: Top vs Bottom Quartile Analysis

* Divide countries into quartiles based on Happiness score.

* Compare the average GDP per capita and average Social support between the top quartile (happiest 25%) and the bottom quartile (least happy 25%).

* What patterns do you notice?

In [254]:
df['Happiness_Quartile'] = pd.qcut(df['Happiness score'], q=4, labels=["Q1 (Lowest 25%)", "Q2", "Q3", "Q4 (Top 25%)"])


In [255]:
top_quartile = df[df['Happiness_Quartile'] == 'Q4 (Top 25%)']
bottom_quartile = df[df['Happiness_Quartile'] == "Q1 (Lowest 25%)"]

In [256]:
top_avg = top_quartile[['Explained by: GDP per capita', 'Explained by: Social support']].mean()
bottom_avg = bottom_quartile[['Explained by: GDP per capita', 'Explained by: Social support']].mean()

In [257]:
comparison = pd.DataFrame({
    "Top Quartile (Happiest 25%)": top_avg,
    "Bottom Quartile (Least Happy 25%)": bottom_avg
})

print(comparison)

                              Top Quartile (Happiest 25%)  \
Explained by: GDP per capita                  1868.810811   
Explained by: Social support                  1162.027027   

                              Bottom Quartile (Least Happy 25%)  
Explained by: GDP per capita                        1027.054054  
Explained by: Social support                         621.621622  


## Q10: Relationship Check with Correlation

* Compute the pairwise correlation coefficients between Happiness score and each of the “Explained by” factors.

* Identify which factor has the strongest positive correlation with happiness.

In [259]:
cols = ['Happiness score', 'Explained by: GDP per capita', 
        'Explained by: Social support', 
        'Explained by: Healthy life expectancy', 
        'Explained by: Freedom to make life choices', 
        'Explained by: Generosity', 
        'Explained by: Perceptions of corruption']

corr_matrix = df[cols].corr()

happiness_corr = corr_matrix['Happiness score'].drop('Happiness score')
print(happiness_corr)

Explained by: GDP per capita                  0.763677
Explained by: Social support                  0.777889
Explained by: Healthy life expectancy         0.740260
Explained by: Freedom to make life choices    0.624822
Explained by: Generosity                      0.063785
Explained by: Perceptions of corruption       0.416216
Name: Happiness score, dtype: float64
